In [1]:
!pip install wandb -q

In [ ]:
import pandas as pd
import random
import numpy as np
import torch
import torch.nn as nn
import wandb
import os
import re
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader



In [3]:
from google.colab import userdata
wandb.login(userdata.get('WANDB_TOKEN'))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pas-vasiljev (pas-vasiljev-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
device

'cuda:0'

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [83]:
df = pd.read_csv('/content/drive/MyDrive/OGO/gp/reviews.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [84]:
def clean_review(text):
    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [85]:
df["review_clean"] = df["review"].apply(clean_review)

df["n_words"] = df["review_clean"].apply(lambda x: len(x.split()))

df.head()

,review,sentiment,review_clean,n_words
0,One of the other reviewers has mentioned that ...,positive,One of the other reviewers has mentioned that ...,304
1,A wonderful little production. <br /><br />The...,positive,A wonderful little production. The filming tec...,156
2,I thought this was a wonderful way to spend ti...,positive,I thought this was a wonderful way to spend ti...,164
3,Basically there's a family where a little boy ...,negative,Basically there's a family where a little boy ...,135
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"Petter Mattei's ""Love in the Time of Money"" is...",225


In [86]:
df["label"] = df["sentiment"].map({"negative": 0, "positive": 1})
df.head()

,review,sentiment,review_clean,n_words,label
0,One of the other reviewers has mentioned that ...,positive,One of the other reviewers has mentioned that ...,304,1
1,A wonderful little production. <br /><br />The...,positive,A wonderful little production. The filming tec...,156,1
2,I thought this was a wonderful way to spend ti...,positive,I thought this was a wonderful way to spend ti...,164,1
3,Basically there's a family where a little boy ...,negative,Basically there's a family where a little boy ...,135,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"Petter Mattei's ""Love in the Time of Money"" is...",225,1


https://habr.com/ru/companies/wunderfund/articles/845272/

In [87]:
class BertDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.target = self.df["label"]
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        X = self.df.loc[idx, "review_clean"]
        y = self.target.loc[idx]

        inputs = self.tokenizer(
            X,
            pad_to_max_length=True,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            max_length=self.max_length,
        )

        x = {
            "ids": torch.tensor(inputs["input_ids"], dtype=torch.long),
            "mask": torch.tensor(inputs["attention_mask"], dtype=torch.long)
        }

        y = torch.tensor(y, dtype=torch.long)

        return x, y

In [88]:
pos = df[df["sentiment"] == "positive"].sample(3000, random_state=42)
neg = df[df["sentiment"] == "negative"].sample(3000, random_state=42)

data = pd.concat([pos, neg]).reset_index(drop=True)
data.head()

,review,sentiment,review_clean,n_words,label
0,I don't know how or why this film has a meager...,positive,I don't know how or why this film has a meager...,451,1
1,For a long time it seemed like all the good Ca...,positive,For a long time it seemed like all the good Ca...,133,1
2,Terry Gilliam's and David Peoples' teamed up t...,positive,Terry Gilliam's and David Peoples' teamed up t...,919,1
3,What is there to say about an anti-establishme...,positive,What is there to say about an anti-establishme...,244,1
4,This movie was made only 48 years after the en...,positive,This movie was made only 48 years after the en...,181,1


Использовалась третья домашка майнора ИАД

In [95]:
def train(model, optimizer, train_dataloader, val_dataloader, criterion=torch.nn.CrossEntropyLoss(), num_epochs=5, device=device, params=None):
    run = wandb.init(project="bert_classifier", config=params)

    train_losses, train_accuracies = [], []
    train_precisions, train_recalls, train_f1s = [], [], []
    val_losses, val_accuracies = [], []
    val_precisions, val_recalls, val_f1s = [], [], []


    for epoch in range(num_epochs):
        model.train()
        epoch_train_losses, epoch_train_accuracies = [], []
        train_true, train_pred = [], []

        # Тут главный цикл обучения
        for batch in tqdm(train_dataloader, desc=f"Training {epoch + 1}/{num_epochs}"):
            x, y = batch

            input_ids = x["ids"].to(device)
            attention_mask = x["mask"].to(device)
            y = y.to(device)

            optimizer.zero_grad()

            output = model(input_ids=input_ids, attention_mask=attention_mask)

            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

            epoch_train_losses.append(loss.item())

            pred_cls = output.argmax(dim=1)
            epoch_train_accuracies.append((pred_cls == y).float().mean().item())

            train_true.extend(y.detach().cpu().numpy())
            train_pred.extend(pred_cls.detach().cpu().numpy())

        model.eval()
        epoch_val_losses, epoch_val_accuracies = [], []
        val_true, val_pred = [], []

        # Тут цикл валидации
        with torch.no_grad():
            for batch in tqdm(val_dataloader, desc=f"Validation {epoch + 1}/{num_epochs}"):
                x, y = batch

                input_ids = x["ids"].to(device)
                attention_mask = x["mask"].to(device)
                y = y.to(device)

                output = model(input_ids=input_ids, attention_mask=attention_mask)

                loss = criterion(output, y)
                epoch_val_losses.append(loss.item())

                pred_cls = output.argmax(dim=1)
                epoch_val_accuracies.append((pred_cls == y).float().mean().item())

                val_true.extend(y.detach().cpu().numpy())
                val_pred.extend(pred_cls.detach().cpu().numpy())

        # Дальше просто считаем все меррики
        epoch_loss = np.mean(epoch_train_losses)
        epoch_acc = np.mean(epoch_train_accuracies)
        epoch_precision = precision_score(train_true, train_pred)
        epoch_recall = recall_score(train_true, train_pred)
        epoch_f1 = f1_score(train_true, train_pred)

        val_loss = np.mean(epoch_val_losses)
        val_acc = np.mean(epoch_val_accuracies)
        val_precision = precision_score(val_true, val_pred)
        val_recall = recall_score(val_true, val_pred)
        val_f1 = f1_score(val_true, val_pred)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": epoch_loss,
            "train_accuracy": epoch_acc,
            "train_precision": epoch_precision,
            "train_recall": epoch_recall,
            "train_f1": epoch_f1,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "val_precision": val_precision,
            "val_recall": val_recall,
            "val_f1": val_f1
        })

        res = {
            "epoch": epoch + 1,
            "train_loss": round(epoch_loss, 3),
            "train_acc": round(epoch_acc, 3),
            "train_precision": round(epoch_precision, 3),
            "train_recall": round(epoch_recall, 3),
            "train_f1": round(epoch_f1, 3),
            "val_loss": round(val_loss, 3),
            "val_acc": round(val_acc, 3),
            "val_precision": round(val_precision, 3),
            "val_recall": round(val_recall, 3),
            "val_f1": round(val_f1, 3)
        }

        display(pd.DataFrame([res]))
    # Логируем артефакты
    model_path = "tuned_bert.pt"
    torch.save(model.state_dict(), model_path)
    tokenizer.save_pretrained("tokenizer")

    run_artifact = wandb.Artifact("bert_files", type="experiment")
    run_artifact.add_file("train_df.csv")
    run_artifact.add_file("test_df.csv")
    run_artifact.add_file(model_path)
    run_artifact.add_dir("tokenizer")
    wandb.log_artifact(run_artifact)

    wandb.finish()

In [96]:
class BertClassifier(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model)
        self.dropout = nn.Dropout(0.4)

        self.classifier = self.classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):

        outputs = self.bert(input_ids, attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]

        output = self.dropout(cls)
        output = self.classifier(output)
        return output

In [97]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = BertClassifier(model_name).to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [98]:
train_df, test_df = train_test_split(data, test_size=0.2, random_state=42, stratify=data["label"])

train_df.to_csv("train_df.csv", index=False)
test_df.to_csv("test_df.csv", index=False)

train_dataset = BertDataset(train_df, tokenizer, max_length=512)
test_dataset = BertDataset(test_df, tokenizer, max_length=512)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [99]:
model = BertClassifier(model_name).to(device)
epochs = 5
lr = 3e-6
weight_decay = 0.04
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

params = {
    "model_name": model_name,
    "num_classes": 2,
    "max_length": 512,
    "batch_size": 32,
    "epochs": epochs,
    "learning_rate": lr,
    "weight_decay": weight_decay,
    "optimizer": "AdamW",
    "loss": "CrossEntropyLoss",
    "device": device
}

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [100]:
train(
    model=model,
    optimizer=optimizer,
    train_dataloader=train_loader,
    val_dataloader=test_loader,
    criterion=criterion,
    num_epochs=epochs,
    device=device,
    params=params
)

Training 1/5:   0%|          | 0/150 [00:00<?, ?it/s]

Validation 1/5:   0%|          | 0/38 [00:00<?, ?it/s]

,epoch,train_loss,train_acc,train_precision,train_recall,train_f1,val_loss,val_acc,val_precision,val_recall,val_f1
0,1,0.673,0.586,0.598,0.528,0.561,0.561,0.807,0.911,0.68,0.779


Training 2/5:   0%|          | 0/150 [00:00<?, ?it/s]

Validation 2/5:   0%|          | 0/38 [00:00<?, ?it/s]

...

Training 5/5:   0%|          | 0/150 [00:00<?, ?it/s]

Validation 5/5:   0%|          | 0/38 [00:00<?, ?it/s]

,epoch,train_loss,train_acc,train_precision,train_recall,train_f1,val_loss,val_acc,val_precision,val_recall,val_f1
0,5,0.171,0.941,0.943,0.939,0.941,0.279,0.899,0.867,0.942,0.903


wandb: Adding directory to artifact (tokenizer)... Done. 0.0s


epoch,▁▃▅▆█
train_accuracy,▁▇▇██
train_f1,▁▇▇██
train_loss,█▄▂▁▁
train_precision,▁▇▇██
train_recall,▁▆▇██
val_accuracy,▁▇▇██
val_f1,▁▇▇██
val_loss,█▁▁▁▁
val_precision,▇▇█▃▁
+1,...
